### 1. Comprovació de Connectivitat amb AWS RDS (PostgreSQL)

Abans de carregar dades, entrenar models o obrir connexions pesades, aquest bloc fa un test de xarxa de baix nivell. És una xarxa de seguretat que ens confirma que l'ordinador (o el servidor on s'executa el notebook) té una ruta directa i oberta cap a la base de dades d'AWS.

**Passos clau de la comprovació:**
* **Càrrega de variables d'entorn (`load_dotenv`)**: Prepara el terreny per llegir credencials segures des d'un fitxer `.env` als passos següents, evitant exposar contrasenyes en el codi.
* **Resolució DNS (`socket.gethostbyname`)**: Verifica que l'URL del clúster d'AWS RDS es pot traduir correctament a una adreça IP. Si això falla, normalment vol dir que no tenim internet o l'URL té un error tipogràfic.
* **Test de Port TCP (`sock.connect_ex`)**: Fa un "ping" silenciós específicament al port `5432` (el port estàndard de PostgreSQL). Si la resolució DNS funciona però aquest pas falla, el problema gairebé segur és de permisos: la nostra IP no està a la llista blanca (*Inbound rules*) del *Security Group* d'AWS.

Aquesta funció ens estalvia hores de depuració en indicar-nos immediatament si hi ha un tallafocs bloquejant el pas abans i tot de posar l'usuari i la contrasenya.

In [25]:
import socket
import psycopg2
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

# Test de conectividad básica
def test_connectivity():
    host = 'database-1.cveau0o428yi.us-east-1.rds.amazonaws.com'
    port = 5432

    try:
        # Test de resolución DNS
        ip = socket.gethostbyname(host)
        print(f"✓ DNS resuelto: {host} -> {ip}")

        # Test de conexión TCP
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(5)
        result = sock.connect_ex((host, port))
        sock.close()

        if result == 0:
            print("✓ Puerto 5432 está accesible")
            return True
        else:
            print(f"✗ Puerto 5432 NO accesible (código error: {result})")
            return False
    except Exception as e:
        print(f"✗ Error de conectividad: {e}")
        return False

test_connectivity()

✓ DNS resuelto: database-1.cveau0o428yi.us-east-1.rds.amazonaws.com -> 34.232.252.216
✓ Puerto 5432 está accesible


True

### 2. Extracció de Dades des de PostgreSQL (AWS RDS)

Un cop confirmada la connectivitat, aquest bloc s'encarrega d'anar a buscar les dades històriques reals i portar-les a la memòria del *notebook* per poder començar a treballar. Utilitza la llibreria `psycopg2` per comunicar-se amb la base de dades i `pandas` per estructurar la informació.

**Passos clau de l'extracció:**
* **Gestió de Credencials:** Carrega l'usuari i la contrasenya des de les variables d'entorn (`.env`). Això és una bona pràctica de seguretat fonamental per no pujar mai contrasenyes en clar al repositori de codi (com GitHub).
* **Connexió Segura:** S'estableix la connexió indicant `sslmode='require'`, la qual cosa garanteix que les dades viatgen xifrades des dels servidors d'AWS fins al nostre entorn.
* **La Consulta SQL i l'Encreuament de Taules (JOIN):** La consulta selecciona únicament les columnes (característiques o *features*) que el nostre model necessita. Mitjançant l'clàusula `INNER JOIN`, unim la taula `INCIDENCE` amb la taula `CHARGER` a través del `charger_id`. Això crea una taula mestra on cada fila té la informació del tiquet de manteniment juntament amb totes les característiques tècniques del carregador afectat.
* **Càrrega a Pandas:** La funció `pd.read_sql` fa la màgia d'executar la consulta i convertir directament el resultat en un `DataFrame` de Pandas (`df`), tancant la connexió de xarxa just després per alliberar recursos.

In [26]:
import psycopg2
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

password = os.environ.get('DB_PASSWORD')
user = os.environ.get('DB_USER')
print(password)
def get_prediction_data():
    """Query data and return as pandas DataFrame"""

    conn = psycopg2.connect(
        host='database-1.cveau0o428yi.us-east-1.rds.amazonaws.com',
        port=5432,
        database='postgres',
        user=user,
        password=password,
        sslmode='require'
    )

    query = """
    SELECT
        -- CHARGER features
        c.brand,
        c.model,
        c.number_of_plugs,
        c.connector_types,
        c.has_rfid,
        c.environment,
        c.power_type,
        c.phase_type,
        c.max_power_kw,
        c.ocpp_version,
        c.telecom_provider,
        c.installation_date,

        -- INCIDENCE features
        i.priority,
        i.description,
        i.estimated_duration_min

    FROM INCIDENCE i
    INNER JOIN CHARGER c ON i.charger_id = c.charger_id
    """

    df = pd.read_sql(query, conn)
    conn.close()

    return df

# Get the data
df = get_prediction_data()

# Show result
print(f"Loaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")
print("\nFirst 5 rows:")
print(df.head())

postgres
Loaded 500 rows
Columns: ['brand', 'model', 'number_of_plugs', 'connector_types', 'has_rfid', 'environment', 'power_type', 'phase_type', 'max_power_kw', 'ocpp_version', 'telecom_provider', 'installation_date', 'priority', 'description', 'estimated_duration_min']

First 5 rows:
     brand  model  number_of_plugs connector_types  has_rfid environment  \
0  Wallbox  Bàsic                5         CHAdeMO     False       EdRSR   
1  Wallbox  Bàsic                5         CHAdeMO     False       EdRSR   
2  Wallbox  Bàsic                5         CHAdeMO     False       EdRSR   
3  Wallbox  Bàsic                5         CHAdeMO     False       EdRSR   
4  Wallbox  Bàsic                5         CHAdeMO     False       EdRSR   

  power_type    phase_type  max_power_kw ocpp_version telecom_provider  \
0      Mixta  No aplica/DC          22.0     OCPP 1.5         Vodafone   
1      Mixta  No aplica/DC          22.0     OCPP 1.5         Vodafone   
2      Mixta  No aplica/DC        

/tmp/ipykernel_4454/159319982.py:48: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


### 3. Processament de Text Lliure amb TF-IDF

Aquest bloc s'encarrega de traduir la columna de text lliure `description` a un format numèric que l'algoritme XGBoost pugui entendre. Per fer-ho, utilitza la tècnica **TF-IDF (Term Frequency - Inverse Document Frequency)**, que no només compta quantes vegades apareix una paraula, sinó que li dóna més pes a les paraules clau rares i rellevants (com "curtcircuit" o "trencat") i menys pes a les paraules molt comunes (com "el", "la", "a").



**Passos clau del processament de text:**
* **Tractament de nuls (`fillna`)**: És vital canviar els valors absents (`NaN`) per cadenes de text buides (`''`). Si no ho fem, el vectoritzador donaria un error en intentar processar un "no-text".
* **Configuració del Vectoritzador (`TfidfVectorizer`)**:
    * `max_features=100`: Limita l'aprenentatge a les 100 paraules o frases més importants. Això és crucial per no generar una matriu gegantina i dispersa (amb milers de columnes plenes de zeros) que faria que l'XGBoost anés molt lent i perdés precisió.
    * `ngram_range=(1, 2)`: Permet que el model no només aprengui paraules soltes (unigrames, ex: "pantalla"), sinó també combinacions de dues paraules contigües (bigrames, ex: "pantalla trencada").
* **Transformació Matemàtica (`fit_transform`)**: Llegeix totes les descripcions, aprèn el diccionari de les 100 expressions més rellevants i calcula la puntuació de cada paraula per a cada fila.
* **Integració al DataFrame (`pd.concat`)**: Converteix la matriu matemàtica resultant en noves columnes de Pandas, afegint-hi el prefix `desc_tfidf_` per identificar-les clarament. Finalment, s'ajunta amb la resta de dades i s'elimina la columna de text original, deixant el dataset a punt per als següents passos numèrics.

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Tractament de valors nuls i preparació del text
df['description'] = df['description'].fillna('')

# 2. Inicialitzar el TF-IDF
# Limitem 'max_features' (ex: 100-300) perquè XGBoost no es saturi amb una matriu massa dispersa.
# NOTA: Ajusta 'stop_words' a l'idioma de les teves dades (pots passar una llista personalitzada si és en català/castellà).
tfidf = TfidfVectorizer(max_features=100, ngram_range=(1, 2))

# 3. Ajustar i transformar la descripció
tfidf_matrix = tfidf.fit_transform(df['description'])

# 4. Crear un DataFrame amb les noves columnes (amb prefix per identificar-les)
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=[f"desc_tfidf_{word}" for word in tfidf.get_feature_names_out()],
    index=df.index # Molt important per no desquadrar les files al concatenar
)

# 5. Concatenar amb el DataFrame original i eliminar la columna de text original
df = pd.concat([df.drop(columns=['description']), tfidf_df], axis=1)

print(df.head())

     brand  model  number_of_plugs connector_types  has_rfid environment  \
0  Wallbox  Bàsic                5         CHAdeMO     False       EdRSR   
1  Wallbox  Bàsic                5         CHAdeMO     False       EdRSR   
2  Wallbox  Bàsic                5         CHAdeMO     False       EdRSR   
3  Wallbox  Bàsic                5         CHAdeMO     False       EdRSR   
4  Wallbox  Bàsic                5         CHAdeMO     False       EdRSR   

  power_type    phase_type  max_power_kw ocpp_version  ... installation_date  \
0      Mixta  No aplica/DC          22.0     OCPP 1.5  ...        2016-05-09   
1      Mixta  No aplica/DC          22.0     OCPP 1.5  ...        2016-05-09   
2      Mixta  No aplica/DC          22.0     OCPP 1.5  ...        2016-05-09   
3      Mixta  No aplica/DC          22.0     OCPP 1.5  ...        2016-05-09   
4      Mixta  No aplica/DC          22.0     OCPP 1.5  ...        2016-05-09   

             priority estimated_duration_min  desc_tfidf_entre

### 4. Preprocessament de Variables Estructurades

Els models de *Machine Learning* basats en arbres com l'XGBoost són molt potents, però tenen un requisit estricte: **només entenen números**. Aquest bloc de codi s'encarrega de traduir cada tipus de dada categòrica o temporal a un format numèric òptim segons la seva naturalesa matemàtica.

**Passos clau del preprocessament:**
* **Variables Booleanes (`has_rfid`, `active`)**: La transformació més senzilla. Es converteixen els valors `True`/`False` a `1` i `0` utilitzant `.astype(int)`.
* **Variables Ordinals (`model`, `ocpp_version`)**: Aquestes categories tenen una jerarquia o ordre lògic. Fent un mapeig manual (ex: Bàsic=0, Avançat=1, Ultra=2), preservem aquesta relació de magnitud perquè l'arbre de decisió entengui que un model és "superior" a l'altre.
* **Dates (`installation_date`)**: Els models no saben llegir formats temporals clàssics (DD/MM/YYYY). Extraient només l'`install_year` (l'any d'instal·lació), aconseguim una variable numèrica simple que l'algoritme utilitzarà com a indicador directe de l'antiguitat i el possible desgast del maquinari.
* **Variables Nominals (*One-Hot Encoding*)**: Categories com `brand` o `priority` no tenen cap ordre matemàtic. Si els assignéssim números consecutius (1, 2, 3), el model assumiria erròniament que una marca "val més" que una altra. La funció `pd.get_dummies` soluciona això creant columnes binàries per a cada opció.
    * El paràmetre `drop_first=True` evita la **multicolinealitat**: si un carregador no és de les marques A ni B, el model ja dedueix matemàticament que és de la marca C, estalviant memòria i evitant soroll.
* **Variables Numèriques Pures (`number_of_plugs`, `max_power_kw`)**: Es força la seva conversió numèrica amb `pd.to_numeric` (i `errors='coerce'` per transformar qualsevol text inesperat en `NaN`), assegurant que les magnituds reals com la potència arribin intactes al model.

In [28]:
import pandas as pd
import numpy as np

# 1. Variables Booleanes
# has_rfid: true, false.
df['has_rfid'] = df['has_rfid'].astype(int)

# 2. Variables Ordinals
# model: "Bàsic", "Avançat", "Ultra-ràpid".
model_mapping = {
    "Bàsic": 0,
    "Avançat": 1,
    "Ultra-ràpid": 2
}
df['model'] = df['model'].map(model_mapping)

# ocpp_version: "OCPP 1.5", "OCPP 1.6", "OCPP 2.0.1".
ocpp_mapping = {
    "OCPP 1.5": 0,
    "OCPP 1.6": 1,
    "OCPP 2.0.1": 2
}
df['ocpp_version'] = df['ocpp_version'].map(ocpp_mapping)

# 3. Dates (Calcular l'antiguitat del carregador)
# Convertim a format datetime
# installation_date: 2015/01/01 - actualitat
df['installation_date'] = pd.to_datetime(df['installation_date'], format='%Y/%m/%d', errors='coerce')
df['install_year'] = df['installation_date'].dt.year
df = df.drop(columns=['installation_date'])

# 4. Variables Nominals (One-Hot Encoding). Permet evitar que el model busqui relacions matematiques on no n'hi ha.
nominals = [
    # brand: "Circutor", "Wallbox", "Kempower".
    'brand',
    # connector_types: "Tipus 2", "CCS2", "CHAdeMO".
    'connector_types',
    # environment: "EdRSR" , "EdRR" , "EdRUR".
    'environment',
    # power_type: "AC", "DC", "Mixta".
    'power_type',
    # phase_type: "Monofásico" , "Trifásico", "No aplica/DC".
    'phase_type',
    # telecom_provider: "Movistar", "Vodafone", "Orange".
    'telecom_provider',
    # priority: "Correctiu crític" , "Correctiu no crític" , "Manteniment preventiu programat",
    #           "Posada en marxa", "Visita de diagnosi".
    'priority'
]

# drop_first=True. Si obtenim 3 columnes booleanes, amb saber el valor de les dues primeres la tercera es deduible
df = pd.get_dummies(df, columns=nominals, drop_first=True, dtype=int)

# 5. Variables Numèriques Pures
# number_of_plugs: 1 - 5
df['number_of_plugs'] = pd.to_numeric(df['number_of_plugs'], errors='coerce')
# max_power_kw: 3.7 - 400
df['max_power_kw'] = pd.to_numeric(df['max_power_kw'], errors='coerce')

### 5. Preparació Final de Variables i Divisió Train/Test

Abans d'alimentar l'algoritme, necessitem estructurar les dades tal com ho requereix l'estàndard del *Machine Learning* supervisat: separant les "preguntes" (característiques) de les "solucions" (duració real) i amagant una part de les dades per fer un examen final al model.



**Passos clau d'aquesta fase:**
* **Neteja de la Variable Objectiu (`Target`)**: Eliminem qualsevol registre on la `duracio_estimada` sigui nul·la. L'algoritme no pot aprendre a predir el temps d'una reparació si històricament no se li va registrar quant va trigar.
* **Separació d'Entrada (X) i Sortida (y)**:
    * `X` (Matriu de característiques): Conté tota la informació prèvia del carregador i l'avaria.
    * `y` (Vector objectiu): Conté únicament els minuts/hores que volem predir.
* **Preservació de l'Estructura (`joblib.dump`)**: Aquest és el pas més crític per a la **posada en producció**. En exportar la llista exacta de columnes de la `X` a un fitxer `.pkl`, garantim que quan arribi un tiquet nou a l'API, puguem reconstruir exactament la mateixa estructura de columnes (incloent-hi els zeros de les categories que faltin) perquè l'XGBoost no doni error.
* **Divisió d'Aprenentatge i Avaluació (`train_test_split`)**:
    * Separem el dataset: un 80% (`Train`) perquè el model estudiï i busqui patrons, i un 20% (`Test`) que "amaguem" per avaluar-lo un cop entrenat.
    * El paràmetre `random_state=42` garanteix que la barreja de dades sigui exactament igual cada vegada que executem el codi, fent que els resultats siguin completament reproduïbles.
* **Nota sobre el codi comentat**: S'inclou la configuració bàsica de l'`XGBRegressor` i la funció `.fit()` com a referència anatòmica d'un entrenament simple, però es deixa comentat per donar pas a la cerca avançada dels millors paràmetres (Hyperparameter Tuning).

In [29]:
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBRegressor
import joblib

# 1. Netejar nuls de la variable a predir
df = df.dropna(subset=['estimated_duration_min'])
# Treure la duracio_estimada dels features, ja que es el valor de sortida, no d'entrada
X = df.drop(columns=['estimated_duration_min'])
y = df['estimated_duration_min']

# 2. Guardar les columnes per a futures prediccions
columnes_entrenament = X.columns.tolist()
joblib.dump(columnes_entrenament, 'columnes_xgboost.pkl')

# 3. Dividir en dades d'entrenament (80%) i test (20%). Random_state barreja les dades.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

""" 4. Configuracio entrenament XGBoost basica
model_xgb = XGBRegressor(
    n_estimators=100,      # Nombre d'arbres
    learning_rate=0.1,     # Velocitat d'aprenentatge
    max_depth=6,           # Profunditat màxima de cada arbre
    random_state=42        # Fixar la llavor perquè sigui reproduïble
)
# Iniciar entrenament
model_xgb.fit(X_train, y_train)
print("Model entrenat amb èxit!")
"""



' 4. Configuracio entrenament XGBoost basica\nmodel_xgb = XGBRegressor(\n    n_estimators=100,      # Nombre d\'arbres\n    learning_rate=0.1,     # Velocitat d\'aprenentatge\n    max_depth=6,           # Profunditat màxima de cada arbre\n    random_state=42        # Fixar la llavor perquè sigui reproduïble\n)\n# Iniciar entrenament\nmodel_xgb.fit(X_train, y_train)\nprint("Model entrenat amb èxit!")\n'

### 6. Optimització d'Hiperparàmetres (Hyperparameter Tuning)

En lloc de configurar l'XGBoost a cegues, aquest bloc utilitza la tècnica de **Cerca en Quadrícula (Grid Search)**. Automatitza la creació de desenes de models diferents per avaluar quin d'ells té la configuració exacta que prediu millor la duració dels manteniments.



**Conceptes clau d'aquest procés:**
* **La Graella d'Opcions (`param_grid`)**: Es defineix un diccionari amb els diferents valors a provar. Aquí combinem la quantitat d'arbres (`n_estimators`), la velocitat d'aprenentatge (`learning_rate`) i la complexitat de l'arbre (`max_depth`). El sistema calcularà totes les combinacions possibles (en aquest cas, $3 \times 3 \times 3 = 27$ configuracions diferents).
* **Validació Creuada (`cv=3`)**: Per evitar que un model tregui bona nota per pura "sort" en memoritzar les dades, el cercador parteix el grup d'entrenament en 3 blocs. Cada combinació s'avalua 3 vegades amb trossos de dades diferents. En total, l'ordinador entrenarà en segon pla $27 \times 3 = 81$ models XGBoost.
* **Mètrica d'Avaluació (`scoring`)**: Utilitzem el `neg_mean_absolute_error` (Error Absolut Mitjà). Li estem dient al cercador: *"El model guanyador serà aquell que, de mitjana, s'equivoqui menys minuts respecte a la realitat"*.
* **Processament en Paral·lel (`n_jobs=-1`)**: Atès que entrenar 81 models és una tasca pesada, aquesta instrucció ordena a l'ordinador que utilitzi el 100% dels nuclis del processador al mateix temps per acabar molt més ràpid.
* **Extracció del Guanyador (`best_estimator_`)**: Un cop acabat el torneig, la funció guarda automàticament a la variable `model_xgb_optim` la versió de l'algoritme que ha demostrat ser més precisa. Aquest serà el nostre "cervell" definitiu per a la posada en producció.

In [30]:
# 1. Definim el menú d'opcions que volem que el bucle provi
param_grid = {
    'n_estimators': [50, 100, 200],     # Provarà amb pocs, normals i molts arbres
    'learning_rate': [0.01, 0.05, 0.1], # Provarà diferents velocitats d'aprenentatge
    'max_depth': [3, 5, 7]              # Provarà arbres poc profunds i arbres complexos
}

# 2. Inicialitzem el model "base" (sense paràmetres, només la llavor)
xgb_base = XGBRegressor(random_state=42)

# 3. Configurem el cercador (el super-bucle)
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=3,
    n_jobs=-1,
    verbose=1
)

# 4. Encenem l'entrenament de totes les combinacions
print("Buscant millors parametres.")
grid_search.fit(X_train, y_train)

# 5. Extreure els resultats
print("Recerca finalitzada!")
print(f"Els millors paràmetres trobats són: {grid_search.best_params_}")

# 6. Ens guardem el millor model per fer-lo servir
model_xgb_optim = grid_search.best_estimator_

Buscant millors parametres.
Fitting 3 folds for each of 27 candidates, totalling 81 fits
Recerca finalitzada!
Els millors paràmetres trobats són: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 50}


### 7. Exportació del Model i Artefactes per a Producció (AWS Lambda)

Un cop hem trobat el model òptim, de res serveix si es queda atrapat a la memòria RAM d'aquest *notebook*. Aquest últim pas s'encarrega d'empaquetar el "cervell" de l'algoritme i els seus traductors perquè puguin ser enviats a un servidor al núvol (AWS Lambda) i començar a predir nous tiquets de manteniment en temps real.

**Conceptes clau d'aquesta exportació:**
* **Format Natiu JSON (`save_model`)**: En lloc de fer servir llibreries genèriques de Python, guardem el model fent servir la funció nativa d'XGBoost en format `.json`. Això té dos grans avantatges:
    1. **Lleugeresa i Compatibilitat**: És un fitxer de text pla, extremadament ràpid de carregar a AWS Lambda i que no dóna problemes de versions entre diferents sistemes operatius.
    2. **Incremental Learning (Aprenentatge Continu)**: Aquest format guarda l'estat exacte de cada branca i node. En el futur, quan tinguem milers de tiquets nous, podrem carregar aquest JSON i dir-li a l'algoritme que "afegeixi" arbres nous per aprendre dels errors recents, sense haver de reentrenar tota la història des del 2015.
* **El Traductor de Text (`tfidf_vectorizer.pkl`)**: Guardem el diccionari exacte de les 100 paraules clau que ha après el model i els seus pesos matemàtics. A producció, quan un operari escrigui "pantalla trencada", hem de garantir que el sistema ho tradueixi exactament als mateixos números que va fer servir durant l'entrenament.

> **Resum de Fitxers a Producció:** Juntament amb el `columnes_xgboost.pkl` que hem guardat en passos anteriors, ara ja tenim la **"Trinitat" de la inferència**:
> 1. `model_xgboost_duracio.json` (El motor de predicció)
> 2. `tfidf_vectorizer.pkl` (El traductor de descripcions)
> 3. `columnes_xgboost.pkl` (La plantilla d'estructura de dades)

In [31]:
# 1. Guardar els pesos i arbres d'XGBoost en format natiu
model_xgb_optim.save_model('model_xgboost_duracio.json')

# 2. (Recordatori) Guardar les eines de preprocessament necessàries per Lambda
import joblib
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']

In [32]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# 1. Fer les prediccions sobre el conjunt de test
y_pred = model_xgb_optim.predict(X_test)

# 2. Calcular mètriques
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("📊 RESULTATS DE PRECISIÓ:")
print(f"---------------------------")
print(f"MAE (Error mig): {mae:.2f} minuts")
print(f"RMSE: {rmse:.2f} minuts")
print(f"R² Score: {r2:.4f} (1.0 és perfecte)")

📊 RESULTATS DE PRECISIÓ:
---------------------------
MAE (Error mig): 29.25 minuts
RMSE: 33.69 minuts
R² Score: -0.0166 (1.0 és perfecte)


In [22]:
import pandas as pd
import joblib
from xgboost import XGBRegressor

# 1. Carregar el model
model_predict = XGBRegressor()
model_predict.load_model('model_xgboost_duracio.json')

# 2. Carregar la llista de columnes que el model espera
columnes_model = joblib.load('columnes_xgboost.pkl')

def predir_durada(dades_input):
    """
    dades_input: Pot ser un diccionari amb les claus:
    'brand', 'model', 'priority', 'max_power_kw', 'installation_date', etc.
    """
    # A. Convertir a DataFrame
    df_input = pd.DataFrame([dades_input])

    # B. Pre-processat (Exactament igual que a l'entrenament!)
    # 1. Antiguitat
    df_input['installation_date'] = pd.to_datetime(df_input['installation_date'])
    df_input['charger_age_days'] = (pd.Timestamp.now() - df_input['installation_date']).dt.days

    # 2. Treure el que no serveix
    df_input = df_input.drop(columns=['installation_date', 'description'], errors='ignore')

    # 3. One-Hot Encoding (pd.get_dummies)
    df_input = pd.get_dummies(df_input)

    # 4. RE-ALINEAR COLUMNES (Crucial!)
    # Afegim les columnes que falten (amb valor 0) i eliminem les que sobrin
    df_final = df_input.reindex(columns=columnes_model, fill_value=0)

    # C. Predicció
    prediccio = model_predict.predict(df_final)
    return prediccio[0]

nova_incidencia = {
  'brand': 'Kempower',
  'model': 'Ultra-ràpid',
  'number_of_plugs': 4,
  'connector_types': 'CCS2',
  'has_rfid': True,
  'environment': 'EdRR',
  'power_type': 'DC',
  'phase_type': 'No aplica/DC',
  'max_power_kw': 150.0,
  'ocpp_version': 'OCPP 2.0.1',
  'telecom_provider': 'Vodafone',
  'installation_date': '2023-01-15',
  'priority': 'Correctiu crític',
  'active': True,
  'description': 'No carrega, error intern'
}

minuts_estimats = predir_durada(nova_incidencia)
print(f"⏱️ Predicció de temps de resolució: {minuts_estimats:.2f} minuts")

⏱️ Predicció de temps de resolució: 91.46 minuts


### 8. Aprenentatge Incremental (Manteniment del Model a Futur)

Un cop el teu model estigui en producció a AWS Lambda, la teva base de dades anirà acumulant centenars de tiquets de manteniment nous cada mes. D'aquí a un any, l'ideal és que l'algoritme aprengui d'aquestes noves avaries (especialment si apareixen models nous de carregadors). Aquest bloc demostra com fer-ho de manera eficient sense haver de reprocessar tota la història des del 2015.



**Com funciona aquesta actualització contínua?**
* **La trampa del `.fit()`**: Normalment, quan crides la funció `.fit()` a *Machine Learning*, el model esborra tot el que sabia i comença des de zero. Si només li passes les dades de l'últim mes, oblidaria tot el passat (això es coneix com a *catastrophic forgetting*).
* **El paràmetre màgic (`xgb_model`)**: En afegir `xgb_model='model_xgboost_duracio.json'` a la funció d'entrenament, li estem dient a l'XGBoost: *"Carrega els arbres de decisió que ja tenies guardats, i fes servir aquestes dades noves (`X_nou`, `y_nou`) només per afegir arbres extra que corregeixin els errors recents"*.
* **Sobreescriptura i Desplegament (`save_model`)**: Finalment, desem el model actualitzat picant a sobre del mateix fitxer JSON. Aquest nou fitxer pesa una miqueta més (perquè té els arbres vells + els nous) i ja està a punt per substituir l'antic a la teva funció Lambda d'AWS.

> **L'avantatge de negoci:** Aquest procés d'Aprenentatge Incremental triga segons en lloc de minuts o hores. Et permet automatitzar un procés (per exemple, cada diumenge a la nit) on el model s'actualitza tot sol amb els tiquets de la setmana, estalviant moltíssims costos de computació al núvol.

In [ ]:
# Quan tinguis les noves dades preparades (X_nou, y_nou)
model_actualitzat = XGBRegressor()

# El paràmetre 'xgb_model' li diu que no comenci de zero, sinó que carregui els pesos previs
model_actualitzat.fit(X_nou, y_nou, xgb_model='model_xgboost_duracio.json')

# Sobreescriure el fitxer amb el model actualitzat per enviar-lo a AWS Lambda
model_actualitzat.save_model('model_xgboost_duracio.json')